# Source audit companion
Scope: Jalisco, provisional subordinate paid workers, 2023 Q1–2026 Q2.
See docs/source-audit.md for connection settings, docs/raw-join-audit.md for source-join methods, and docs/gate-a-decision.md for the decision.
The optional live cell executes aggregate SQL in a read-only snapshot. All evidence is aggregate and outputs must remain stripped.


In [ ]:
import json
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
evidence = root / "outputs/source-audit-2026-09-10.json"
raw_evidence = root / "outputs/raw-join-audit-2026-09-10.json"
audit = json.loads(evidence.read_text())
raw_audit = json.loads(raw_evidence.read_text())

In [ ]:
# Counts and denominators remain quarterly, never pooled population estimates.
candidates = [r for r in audit["funnel"] if r["step"] == 6]
candidates

In [ ]:
# Exact-income zero is not zero earnings when a positive ING7C band is present.
[
    (r["anio"], r["trimestre"], 100 * r["zero_weight"] / r["valid_weight_sum"])
    for r in audit["numeric"]
    if r["variable"] == "ingocup" and r["valid_weight_sum"] > 0
]

In [ ]:
# Inspect semantic consistency without exposing person-level records.
audit["semantic_consistency"]

In [ ]:
# Compare official-key safety with the current ETL key by inspected period.
[
    {
        "period": period["period"],
        "candidate_records": period["sdem"]["candidate_records"],
        "official_join_safe": period["official_join_safe"],
        "pipeline_risk_detected": period["current_pipeline_risk_detected"],
    }
    for period in raw_audit["periods"]
]

In [ ]:
# Opt in only after configuring the connection described in docs/source-audit.md.
RUN_LIVE = False
if RUN_LIVE:
    import sys

    sys.path.insert(0, str(root))
    from scripts.audit_source import run_query

    audit = run_query((root / "sql/source_audit.sql").read_text())